# MiniT2I 0.3, controlled experiments

Each experiment starts from the same 0.2 checkpoint, trains the same number of steps on the same shards and changes one thing. Then `avaliar.py` scores it on a fixed prompt set and `cego.py` builds a blind A/B page.

| id | change |
|---|---|
| E0 | baseline, original captions, same steps as the others |
| E1 | new Florence 2 captions (80% of the time) |
| E2 | data cleaning by CLIP scores and sharpness |
| E3 | aspect ratio buckets (needs non square support, later) |
| E4 | timestep distribution, `misto:0.2` against the current logit normal |
| E5 | best combination |
| E6 | fine tuning at 256 x 256 |

Run the setup cell, then **either** the captioning cell (can run on a second Colab account) **or** one experiment cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
PASTA = '/content/drive/MyDrive/minit2i'
DADOS, LEGENDAS = f'{PASTA}/dados', f'{PASTA}/legendas'
if os.path.exists('/content/minit2i'):
    !git -C /content/minit2i pull -q
else:
    !git clone -q https://github.com/Samuelpedroso-dev/minit2i /content/minit2i
!pip install -q diffusers sentence-transformers timm einops torchmetrics[image]
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('no HF_TOKEN, everything stays in Drive')
print(len(os.listdir(DADOS)), 'shards,', len(os.listdir(LEGENDAS)) if os.path.exists(LEGENDAS) else 0, 'captioned')

## New captions and quality scores
Original captions are never touched, results go to `minit2i/legendas`. Resumes where it stopped.

In [ ]:
!cd /content/minit2i/geral && python relegendar.py --saida {LEGENDAS} --dados {DADOS} --inicio 0 --fim 40 --minutos 170

## One experiment
All experiments use only the captioned shards and 77 text tokens, so the only difference between them is the change in the table. 3000 steps take a bit over an hour on a T4. Run them after 0.2 finishes, from the same frozen base.

In [ ]:
EXP = 'e1'   # e0, e1, e2, e4, e5
PASSOS = 3000
OPCOES = {
    'e0': '',
    'e1': f'--legendas {LEGENDAS}',
    'e2': f'--legendas {LEGENDAS} --p_nova 0 --filtro "marca_dagua<0.3,texto<0.4,nitidez>40"',
    'e4': '--tempos misto:0.2',
    'e5': f'--legendas {LEGENDAS} --filtro "marca_dagua<0.3,texto<0.4,nitidez>40"',
}[EXP]
# every experiment starts from the same frozen copy of the 0.2 checkpoint
BASE = f'{PASTA}/exp/base.pt'
if not os.path.exists(BASE):
    from huggingface_hub import hf_hub_download, HfApi
    os.makedirs(f'{PASTA}/exp', exist_ok=True)
    u = HfApi().whoami()['name']
    !cp {hf_hub_download(f'{u}/minit2i-treino', 'geral.pt', local_dir='/content/base')} {BASE}
!cd /content/minit2i/geral && python treinar.py --dados {DADOS} --ckpt {PASTA}/exp/{EXP}.pt \
    --restringir {LEGENDAS} --maxlen 77 --iniciar_de {BASE} --sem_site --hf_treino minit2i-{EXP} \
    --local /content/ckpt_{EXP} --passos_exp {PASSOS} --decair_frac 0.2 {OPCOES}

## Evaluation

In [ ]:
!cd /content/minit2i/geral && python avaliar.py --ckpt /content/ckpt_{EXP}/geral.pt --saida {PASTA}/avaliacao/{EXP} --fid 5000
from IPython.display import Image, display
display(Image(f'{PASTA}/avaliacao/{EXP}/grade_attributes.png', width=512))

In [ ]:
# blind A/B page, open the html file from Drive in any browser
!cd /content/minit2i/geral && python cego.py {PASTA}/avaliacao/e0 {PASTA}/avaliacao/{EXP} --saida {PASTA}/avaliacao/cego_e0_{EXP}.html